# Advanced: Multi-Strategy Email Field Analysis with PAMOLA.CORE

**Goal**: Master all 3 email analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Domain-focused analysis (corporate vs free email detection)
- **Strategy 2**: Pattern-based identification (name structure detection)
- **Strategy 3**: Privacy risk assessment (uniqueness and re-identification)
- Calculate comprehensive data quality metrics
- Analyze domain concentration and distribution
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of email privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 02_email_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.email import EmailOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 5 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Dataset features:**
- Mix of corporate and free email providers
- Various naming patterns (first.last, firstname, etc.)
- Intentional duplicates for privacy testing
- Domain concentration for linkage analysis

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'email_analysis_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # First names and last names for generation
    first_names = ['John', 'Jane', 'Michael', 'Sarah', 'David', 'Emily', 'Robert', 'Lisa',
                   'James', 'Mary', 'William', 'Patricia', 'Richard', 'Jennifer', 'Charles']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller',
                  'Davis', 'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez']
    
    # Domain categories
    free_domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'icloud.com']
    corporate_domains = ['company.com', 'acme.com', 'techcorp.com', 'industries.com', 'group.com']
    
    # Generate emails with various patterns
    emails = []
    departments = []
    roles = []
    
    for i in range(n_records):
        first = np.random.choice(first_names).lower()
        last = np.random.choice(last_names).lower()
        
        # Randomly choose domain type (60% free, 40% corporate)
        if np.random.random() < 0.6:
            domain = np.random.choice(free_domains)
            dept = 'External'
            role = 'Contractor'
        else:
            domain = np.random.choice(corporate_domains)
            dept = np.random.choice(['Sales', 'Marketing', 'Engineering', 'HR', 'Finance'])
            role = np.random.choice(['Manager', 'Analyst', 'Director', 'Associate', 'Lead'])
        
        # Randomly choose email pattern
        pattern_type = np.random.random()
        if pattern_type < 0.5:
            # first.last@domain
            email = f"{first}.{last}@{domain}"
        elif pattern_type < 0.7:
            # firstname@domain
            email = f"{first}@{domain}"
        elif pattern_type < 0.85:
            # first_last@domain
            email = f"{first}_{last}@{domain}"
        else:
            # firstlast@domain
            email = f"{first}{last}@{domain}"
        
        emails.append(email)
        departments.append(dept)
        roles.append(role)
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'email': emails,
        'department': departments,
        'role': roles,
        'status': np.random.choice(['Active', 'Inactive'], n_records, p=[0.9, 0.1])
    })
    
    # Introduce intentional duplicates (5%)
    duplicate_indices = np.random.choice(df.index, size=int(n_records * 0.05), replace=False)
    for idx in duplicate_indices:
        dup_idx = np.random.choice([i for i in df.index if i != idx])
        df.loc[idx, 'email'] = df.loc[dup_idx, 'email']
    
    # Introduce some invalid emails (2%)
    invalid_indices = np.random.choice(df.index, size=int(n_records * 0.02), replace=False)
    df.loc[invalid_indices, 'email'] = 'invalid-email'
    
    # Introduce some null values (1%)
    null_indices = np.random.choice(df.index, size=int(n_records * 0.01), replace=False)
    df.loc[null_indices, 'email'] = None
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    
    # Handle different data types safely
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == 'object':
        # For string/object columns
        nunique = df[col].nunique()
        null_count = df[col].isnull().sum()
        
        info_parts = [f"{nunique} unique values"]
        if null_count > 0:
            info_parts.append(f"{null_count} nulls")
        
        print(f"  {col:15s} ({dtype_str:10s}): {', '.join(info_parts)}")
        
    elif pd.api.types.is_numeric_dtype(df[col]):
        # For numeric columns
        try:
            min_val = df[col].min()
            max_val = df[col].max()
            
            if pd.api.types.is_integer_dtype(df[col]):
                print(f"  {col:15s} ({dtype_str:10s}): min={min_val}, max={max_val}")
            else:
                print(f"  {col:15s} ({dtype_str:10s}): min={min_val:.2f}, max={max_val:.2f}")
        except:
            print(f"  {col:15s} ({dtype_str:10s}): [unable to compute range]")
    else:
        # Fallback for other types
        print(f"  {col:15s} ({dtype_str:10s}): [mixed or unknown type]")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - All strategies analyze same field but with different focus
   - Default: `"email"` for all strategies
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "email",   # Domain-focused analysis
    "strategy2_field": "email",   # Pattern-based identification
    "strategy3_field": "email"    # Privacy risk assessment
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy email field analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Domain-Focused Analysis

**How to use:**
- Analyzes email domains to identify provider types
- Distinguishes corporate vs free email providers
- Run to execute domain-focused analysis

**Key parameters:**
- `field_name`: Email field to analyze (from FIELD_CONFIG)
- `top_n=20`: Number of top domains to track
- `min_frequency=5`: Minimum occurrences for domain dictionary
- `analyze_privacy_risk=True`: Enable privacy assessment

**What you'll see:**
- Configuration summary
- Progress: validation → domain extraction → frequency analysis → save
- Completion time (2-5 seconds)
- Domain distribution statistics

**Note:** Identifies corporate email concentration and provider diversity

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: DOMAIN-FOCUSED ANALYSIS")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Domain analysis",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = EmailOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    top_n=20,
    min_frequency=5,
    profile_type='email',
    analyze_privacy_risk=True,
    chunk_size=1000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=1,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Field:            {operation_s1.field_name}")
print(f"  Top N domains:    {operation_s1.top_n}")
print(f"  Min frequency:    {operation_s1.min_frequency}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_domain_analysis',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load and display key results
output_files_s1 = sorted(
    list((task_dir / 'strategy1_domain_analysis' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    with open(output_files_s1[0], 'r') as f:
        stats_s1 = json.load(f)
    
    print(f"\n📊 Domain Statistics:")
    print(f"   Unique domains:  {stats_s1.get('unique_domains', 0)}")
    print(f"   Valid emails:    {stats_s1.get('valid_count', 0)}")
    
    if 'top_domains' in stats_s1:
        print(f"\n🔝 Top 5 Domains:")
        for i, (domain, count) in enumerate(list(stats_s1['top_domains'].items())[:5], 1):
            pct = (count / stats_s1.get('valid_count', 1)) * 100
            print(f"   {i}. {domain:20s}: {count:4d} ({pct:5.1f}%)")

## STRATEGY 2: Pattern-Based Identification

**How to use:**
- Detects personal naming patterns in email addresses
- Identifies name-based structures (first.last, firstname, etc.)
- Run to execute pattern-based analysis

**Key parameters:**
- `field_name`: Email field to analyze (from FIELD_CONFIG)
- `top_n=15`: Track fewer domains for pattern focus
- `min_frequency=3`: Lower threshold for pattern detection
- `analyze_privacy_risk=True`: Assess identification risks

**What you'll see:**
- Configuration summary
- Progress: validation → pattern detection → classification → save
- Completion time (2-5 seconds)
- Pattern distribution percentages

**Note:** High pattern percentages indicate personally identifiable information

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: PATTERN-BASED IDENTIFICATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Pattern detection",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = EmailOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    top_n=15,
    min_frequency=3,
    profile_type='email',
    analyze_privacy_risk=True,
    chunk_size=1000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=1,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Field:            {operation_s2.field_name}")
print(f"  Top N domains:    {operation_s2.top_n}")
print(f"  Min frequency:    {operation_s2.min_frequency}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_pattern_analysis',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load and display key results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_pattern_analysis' / 'output').glob('*_stats_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    with open(output_files_s2[0], 'r') as f:
        stats_s2 = json.load(f)
    
    if 'personal_patterns' in stats_s2:
        patterns = stats_s2['personal_patterns']
        print(f"\n👤 Personal Pattern Detection:")
        print(f"   Valid emails analyzed: {patterns.get('total_valid_emails', 0)}")
        
        if 'pattern_percentages' in patterns:
            print(f"\n   Top Patterns:")
            sorted_patterns = sorted(
                patterns['pattern_percentages'].items(),
                key=lambda x: x[1],
                reverse=True
            )
            for pattern, pct in sorted_patterns[:5]:
                if pct > 0:
                    print(f"   • {pattern.replace('_', ' ').title():30s}: {pct:5.1f}%")

## STRATEGY 3: Privacy Risk Assessment

**How to use:**
- Evaluates uniqueness and re-identification risks
- Calculates singleton rates and concentration metrics
- Run to execute privacy-focused analysis

**Key parameters:**
- `field_name`: Email field to analyze (from FIELD_CONFIG)
- `top_n=10`: Focus on most common domains
- `min_frequency=10`: Higher threshold for privacy focus
- `analyze_privacy_risk=True`: Full privacy assessment

**What you'll see:**
- Configuration summary
- Progress: validation → uniqueness analysis → risk scoring → save
- Completion time (2-5 seconds)
- Privacy risk metrics (uniqueness, singletons)

**Note:** High uniqueness ratios indicate direct identifier status

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: PRIVACY RISK ASSESSMENT")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Privacy assessment",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = EmailOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    top_n=10,
    min_frequency=10,
    profile_type='email',
    analyze_privacy_risk=True,
    chunk_size=1000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=1,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Field:            {operation_s3.field_name}")
print(f"  Top N domains:    {operation_s3.top_n}")
print(f"  Min frequency:    {operation_s3.min_frequency}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_privacy_assessment',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load and display key results
privacy_files_s3 = sorted(
    list((task_dir / 'strategy3_privacy_assessment' / 'output').glob('*_privacy_risk_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if privacy_files_s3:
    with open(privacy_files_s3[0], 'r') as f:
        privacy_s3 = json.load(f)
    
    print(f"\n🔒 Privacy Risk Metrics:")
    print(f"   Unique emails:      {privacy_s3.get('unique_emails', 0)}")
    print(f"   Uniqueness ratio:   {privacy_s3.get('uniqueness_ratio', 0):.2%}")
    print(f"   Singleton count:    {privacy_s3.get('singleton_count', 0)}")
    print(f"   Singleton %:        {privacy_s3.get('singleton_percentage', 0):.2f}%")
    print(f"   Risk level:         {privacy_s3.get('risk_level', 'Unknown')}")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and data quality metrics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Data quality comparison (valid rates)
- Strategy-specific insights

**Note:** All strategies analyze same field but provide different insights

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Domain analysis):    {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Pattern detection):  {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Privacy assessment): {elapsed_s3:6.2f}s")
print(f"  Total:                           {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📈 Data Quality Metrics:")
strategies = [
    ('Strategy 1', output_files_s1),
    ('Strategy 2', output_files_s2),
    ('Strategy 3', output_files_s2)  # Use s2 for consistency
]

for name, files in strategies:
    if files:
        with open(files[0], 'r') as f:
            stats = json.load(f)
        print(f"\n  {name}:")
        print(f"    Valid emails:   {stats.get('valid_count', 0):,}")
        print(f"    Valid rate:     {stats.get('valid_percentage', 0):.2f}%")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Performance, effectiveness, data quality
2. **Statistics Data**: Validation results, domain distributions
3. **Visualizations**: Charts displayed inline (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_domain_analysis', 'Strategy 1: Domain-Focused Analysis'),
    ('strategy2_pattern_analysis', 'Strategy 2: Pattern-Based Identification'),
    ('strategy3_privacy_assessment', 'Strategy 3: Privacy Risk Assessment')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"\n✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"\n⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                print("   " + "-" * 60)
                key_metrics = ['total_records', 'total_rows', 'valid_count', 'invalid_count',
                              'valid_percentage', 'unique_domains', 'null_count']
                
                for key in key_metrics:
                    if key in metrics:
                        value = metrics[key]
                        if isinstance(value, float):
                            if 'percentage' in key:
                                print(f"   {key:20s}: {value:.2f}%")
                            else:
                                print(f"   {key:20s}: {value:.2f}")
                        else:
                            print(f"   {key:20s}: {value:,}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Detailed Statistics (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        stat_files = sorted(
            list(output_dir.glob('*_stats_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if stat_files:
            print(f"\n📊 DETAILED STATISTICS: {stat_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(stat_files[0], 'r') as f:
                    stats = json.load(f)
                
                # Display general statistics
                general_stats = [
                    'total_rows', 'null_count', 'null_percentage',
                    'non_null_count', 'valid_count', 'valid_percentage',
                    'invalid_count', 'invalid_percentage', 'unique_domains'
                ]
                
                for key in general_stats:
                    if key in stats:
                        value = stats[key]
                        if isinstance(value, float):
                            if 'percentage' in key:
                                print(f"   {key:25s}: {value:.2f}%")
                            else:
                                print(f"   {key:25s}: {value:.2f}")
                        else:
                            print(f"   {key:25s}: {value:,}")
                
                # Show top domains if available
                if 'top_domains' in stats:
                    print(f"\n   🌐 Top Domains:")
                    top_domains = stats['top_domains']
                    for domain, count in list(top_domains.items())[:5]:
                        percentage = (count / stats.get('valid_count', count)) * 100
                        print(f"      • {domain:25s}: {count:4,} ({percentage:5.2f}%)")
                
                # Show pattern analysis if available (Strategy 2)
                if 'personal_patterns' in stats:
                    print(f"\n   📋 Email Pattern Analysis:")
                    patterns = stats['personal_patterns']
                    if 'pattern_percentages' in patterns:
                        pattern_pcts = patterns['pattern_percentages']
                        for pattern, pct in sorted(pattern_pcts.items(), key=lambda x: x[1], reverse=True)[:5]:
                            pattern_name = pattern.replace('_', ' ').title()
                            print(f"      • {pattern_name:30s}: {pct:5.2f}%")
                
            except Exception as e:
                print(f"   ⚠️  Error reading statistics: {e}")
    
    # 3. Domain Distribution Dictionary (from output/)
    if output_dir and output_dir.exists():
        # Look for domains dictionary JSON file
        domain_dict_files = sorted(
            list(output_dir.glob('*_domains_dictionary_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if domain_dict_files:
            print(f"\n📁 DOMAIN DISTRIBUTION: {domain_dict_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(domain_dict_files[0], 'r') as f:
                    domain_data = json.load(f)
                
                print(f"   Total domains           : {domain_data.get('total_domains', 0):,}")
                print(f"   Total emails analyzed   : {domain_data.get('total_emails', 0):,}")
                
                if 'domains' in domain_data:
                    domains_list = domain_data['domains']
                    print(f"\n   Top 10 Domains:")
                    
                    for idx, domain_info in enumerate(domains_list[:10], 1):
                        domain = domain_info.get('domain', 'N/A')
                        count = domain_info.get('count', 0)
                        pct = domain_info.get('percentage', 0)
                        print(f"      {idx:2d}. {domain:25s}: {count:4,} ({pct:5.2f}%)")
                        
            except Exception as e:
                print(f"   ⚠️  Error reading domain distribution: {e}")
    
    # 4. Privacy Risk Assessment (from output/ - Strategy 3 specific)
    if 'privacy' in dir_name and output_dir and output_dir.exists():
        risk_files = sorted(
            list(output_dir.glob('*_privacy_risk_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if risk_files:
            print(f"\n🔒 PRIVACY RISK ASSESSMENT: {risk_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(risk_files[0], 'r') as f:
                    risk_data = json.load(f)
                
                # Main risk metrics
                print(f"   Field analyzed          : {risk_data.get('field_name', 'N/A')}")
                print(f"   Total valid emails      : {risk_data.get('total_valid_emails', 0):,}")
                print(f"   Unique emails           : {risk_data.get('unique_emails', 0):,}")
                print(f"   Uniqueness ratio        : {risk_data.get('uniqueness_ratio', 0):.4f}")
                print(f"   Risk Level              : {risk_data.get('risk_level', 'N/A')}")
                
                # Frequency analysis
                if 'most_frequent_count' in risk_data:
                    print(f"\n   📊 Frequency Analysis:")
                    print(f"      Most frequent count     : {risk_data['most_frequent_count']:,}")
                    print(f"      Singleton count         : {risk_data.get('singleton_count', 0):,}")
                    print(f"      Singleton percentage    : {risk_data.get('singleton_percentage', 0):.2f}%")
                
                # Most frequent examples
                if 'most_frequent_examples' in risk_data:
                    print(f"\n   🔝 Most Frequent Emails:")
                    freq_examples = risk_data['most_frequent_examples']
                    
                    for idx, (email, count) in enumerate(list(freq_examples.items())[:10], 1):
                        print(f"      {idx:2d}. {email:30s}: {count:3,} occurrences")
                        
            except Exception as e:
                print(f"   ⚠️  Error reading privacy risk: {e}")
    
    # 5. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Comprehensive Privacy Score

**How to use:**
- Run AFTER all strategies complete
- Calculates multi-dimensional privacy score

**What you'll see:**
- Overall privacy score (0-100)
- Component scores:
  - Domain diversity (concentration risk)
  - Pattern exposure (name-based identification)
  - Uniqueness risk (singleton rate)
  - Provider distribution (corporate vs free)

**Privacy targets:**
- Score ≥ 70: Good privacy protection
- Score ≥ 50: Moderate protection
- Score < 50: High re-identification risk

**Note:** Requires all 3 strategies to complete successfully

In [ ]:
print("\n" + "=" * 80)
print("🔒 COMPREHENSIVE PRIVACY SCORE")
print("=" * 80 + "\n")

try:
    # Load statistics from all strategies
    with open(output_files_s1[0], 'r') as f:
        stats_s1 = json.load(f)
    with open(output_files_s2[0], 'r') as f:
        stats_s2 = json.load(f)
    with open(privacy_files_s3[0], 'r') as f:
        privacy_s3 = json.load(f)
    
    # 1. Domain diversity score (higher unique domains = better)
    unique_domains = stats_s1.get('unique_domains', 0)
    valid_count = stats_s1.get('valid_count', 1)
    domain_diversity = min(100, (unique_domains / valid_count * 100) * 10)  # Scale appropriately
    
    # 2. Pattern exposure score (lower name patterns = better)
    patterns = stats_s2.get('personal_patterns', {})
    pattern_pcts = patterns.get('pattern_percentages', {})
    max_pattern_pct = max(pattern_pcts.values()) if pattern_pcts else 0
    pattern_score = max(0, 100 - max_pattern_pct)
    
    # 3. Uniqueness risk score (lower uniqueness = better)
    uniqueness_ratio = privacy_s3.get('uniqueness_ratio', 0)
    uniqueness_score = max(0, 100 - (uniqueness_ratio * 100))
    
    # 4. Singleton rate score (fewer singletons = better)
    singleton_pct = privacy_s3.get('singleton_percentage', 0)
    singleton_score = max(0, 100 - singleton_pct)
    
    # Overall privacy score (weighted average)
    overall_score = (
        domain_diversity * 0.2 +
        pattern_score * 0.3 +
        uniqueness_score * 0.3 +
        singleton_score * 0.2
    )
    
    print(f"🎯 OVERALL PRIVACY SCORE: {overall_score:.1f}/100")
    print(f"\n📊 Component Scores:")
    print(f"   Domain Diversity:      {domain_diversity:.1f}/100 (concentration risk)")
    print(f"   Pattern Exposure:      {pattern_score:.1f}/100 (name identification)")
    print(f"   Uniqueness Risk:       {uniqueness_score:.1f}/100 (email uniqueness)")
    print(f"   Singleton Rate:        {singleton_score:.1f}/100 (single occurrences)")
    
    print(f"\n💡 Privacy Assessment:")
    if overall_score >= 70:
        print("   ✅ Good - Email field has reasonable privacy protection")
    elif overall_score >= 50:
        print("   ⚠️  Moderate - Consider masking before external sharing")
    else:
        print("   ❌ High Risk - Strong re-identification potential")
        print("      Recommendation: Hash or tokenize email column")
        
except Exception as e:
    print(f"⚠️  Error calculating privacy score: {e}")
    print("   Ensure all 3 strategies completed successfully")

## Step 7: Export Analysis Results

**How to use:**
- Run AFTER all strategies complete
- Exports comprehensive analysis reports

**What you'll see (per strategy):**
- Export confirmation with file path
- Summary statistics
- Key findings preview

**Output structure:**
```
advanced_output/
├── strategy1_domain_analysis/
│   ├── output/*.json
│   ├── dictionaries/*.csv
│   └── visualizations/*.png
├── strategy2_pattern_analysis/
│   └── ...
└── strategy3_privacy_assessment/
    └── ...
```

**Note:** All files include timestamps for version tracking

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS")
print("=" * 80)

print(f"\n📂 Export base directory: {task_dir}\n")

# Summary for each strategy
strategies = [
    ('STRATEGY 1: DOMAIN-FOCUSED ANALYSIS', 'strategy1_domain_analysis', output_files_s1),
    ('STRATEGY 2: PATTERN-BASED IDENTIFICATION', 'strategy2_pattern_analysis', output_files_s2),
    ('STRATEGY 3: PRIVACY RISK ASSESSMENT', 'strategy3_privacy_assessment', output_files_s2)
]

for title, dir_name, output_files in strategies:
    print("=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    strategy_dir = task_dir / dir_name
    
    if output_files:
        print(f"\n✅ Statistics: {output_files[0].name}")
        print(f"   Location: {strategy_dir / 'output'}")
        
        # Load and show summary
        with open(output_files[0], 'r') as f:
            stats = json.load(f)
        
        print(f"\n📈 Summary:")
        print(f"   Total records:   {stats.get('total_rows', 0):,}")
        print(f"   Valid emails:    {stats.get('valid_count', 0):,}")
        print(f"   Valid rate:      {stats.get('valid_percentage', 0):.2f}%")
        print(f"   Unique domains:  {stats.get('unique_domains', 0)}")
    
    # Check for domain dictionary
    dict_files = list((strategy_dir / 'dictionaries').glob('*.csv')) if (strategy_dir / 'dictionaries').exists() else []
    if dict_files:
        print(f"\n✅ Domain dictionary: {len(dict_files)} file(s)")
    
    # Check for visualizations
    viz_files = list((strategy_dir / 'visualizations').glob('*.png')) if (strategy_dir / 'visualizations').exists() else []
    if viz_files:
        print(f"✅ Visualizations: {len(viz_files)} file(s)")
    
    print()

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 All files saved to: {task_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 ANALYSIS COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Domain concentration analysis
- ✅ Personal pattern identification
- ✅ Privacy risk assessment
- ✅ Comprehensive privacy scoring
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Domain analysis reveals organizational structure
- Pattern detection identifies name-based email structures
- Privacy assessment quantifies re-identification risks
- Combined analysis provides holistic data quality view

**Strategy recommendations:**
- **Use Strategy 1** when: Understanding email provider distribution
- **Use Strategy 2** when: Detecting personally identifiable patterns
- **Use Strategy 3** when: Assessing privacy and uniqueness risks

**Privacy best practices:**
- Hash emails before external sharing
- Mask local-part in analytics (keep domain only)
- Consider domain aggregation for public reports
- Monitor singleton rates as privacy indicators

**Next steps:**
- Test with your own datasets
- Adjust privacy thresholds for your use case
- Deploy to production pipelines
- Integrate with data governance workflows

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)